# RF electrode of a silicon modulator: simulation details

This is the long version of the guide [RF electrode of a silicon modulator](palace_bj2012_electrode.md): every job, the fit and the impedance definitions. Start there if you want the device, the pictures and the result.

This notebook reproduces the metal-only control line of Baehr-Jones *et al.*, "Ultralow drive voltage silicon traveling-wave modulator", *Optics Express* **20**(11), 12014 (2012), [doi:10.1364/OE.20.012014](https://doi.org/10.1364/OE.20.012014), with generic gsim tools. The paper measured $Z = 61 - 5j\,\Omega$, 3.8 dB/cm, $L = 457$ nH/m, $R = 4.9$ k$\Omega$/m and $C = 0.12$ nF/m at 10 GHz. femwell solves the coplanar-strip cross-section locally and Palace checks it in 2D and in 3D; for bulk aluminum the prediction is 1.83 dB/cm, about half of the measurement. The junction that loads this line is in the guide [Phase shifter of a silicon modulator](tcad_bj2012_modulator.md).

Reference values (read by eye from the paper's figures, about one marker of uncertainty) are in `data/bj2012_reference.json`. The numbers this notebook computes are stored in `data/rf_benchmark/bj2012_electrode.json`, and the last cell checks them.

## What the notebook computes

| Step | Question | Tool | Result |
|---|---|---|---|
| 1-2. Reference | What are the mode, impedance and loss of the cross-section, with perfect and with lossy aluminum? | femwell, local | $Z$, $R$, $L$, $C$ and loss against the paper |
| 3-4. Palace set-up | Do the 2D mode solves (1 to 40 GHz) and two 3D straight sections (200 µm and 600 µm, a wave port at each end) mesh and validate? | `gsim.palace`, local; the cloud runs are **off** by default (`RUN_CLOUD`) | Valid Palace configs; results only after the cloud runs |
| 5-7. Post-processing | How do the 3D sections give the line's index, loss and impedance? | scikit-rf, local | De-embedding and fit, tested on synthetic S-parameters, applied to the Palace data |
| 8. Volume metal | Does Palace agree with femwell on the loss of bulk aluminum? | femwell and Palace | Loss at 10 GHz for three treatments of the metal |
| 9. Loaded line | What does the junction do to the electrode? | closed form | $Z$, RF index and RF loss against the paper |

**Requirements:** `pip install "gsim[femwell]"`, and a [GDSFactory+](https://gdsfactory.com) account for the cloud runs.

**Units.** Lengths in µm, frequencies in Hz unless a name says GHz.

In [ ]:
import json
import logging
from pathlib import Path

import gdsfactory as gf
import matplotlib.pyplot as plt
import meshio
import numpy as np
import skfem
from scipy.constants import speed_of_light
from skfem.helpers import cross

from gsim.common.modes import Conductor, propagating_modes
from gsim.common.stack.extractor import Layer, LayerStack
from gsim.common.stack.materials import MaterialProperties
from gsim.common.transmission_line import (
    line_params_from_gamma,
    line_params_from_neff,
    loaded_line_params,
    section_abcd,
)
from gsim.femwell import epsilon_by_region, line_reading, solve_modes
from gsim.femwell.adapter import electrode_current
from gsim.palace import BoundaryModeSim, DrivenSim

# The mesher's advice on air boxes targets other simulations.
logging.getLogger("gsim.palace").setLevel(logging.ERROR)

FREQ_HZ = 10e9  # Hz, the paper's frequency

REF_PATH = next(
    p
    for p in (
        Path("data/bj2012_reference.json"),
        Path("nbs/data/bj2012_reference.json"),
    )
    if p.exists()
)
REF = json.loads(REF_PATH.read_text(encoding="utf-8"))
PAPER = REF["metal_only_line_10GHz"]

## The cross-section

One builder draws the electrode for the 2D and the 3D simulations. Coordinates are in µm: $y$ across the line, $z$ up, the strips along $x$.

The numbers come from the paper's Fig. 1b: strips 50 µm wide and 2 µm thick with a 5 µm gap, 2 µm of buried oxide (BOX) and a 750 Ω·cm handle wafer. The paper's text gives only the BOX, the handle and the top-silicon values; the electrode dimensions are read from the figure. Four more inputs are assumptions, because the paper does not give them:

- The strips' lower face is 1.5 µm above the BOX (the 0.09 + 0.75 + 0.6 µm of slab, lower metal and via add up to 1.44 µm).
- The top silicon under the line is absent.
- No oxide covers the strips: the paper's process description ends with "two levels of aluminum" and mentions no passivation.
- The handle wafer is truncated at 100 µm, and the domain is ±200 µm wide with a 100 µm air layer on top.

The outer wall of the domain is a **perfect conductor** in the femwell solves and in the Palace runs: a closed metallic box, which is also what the wave port's own eigenproblem assumes. The box is a model of the reference solves, not of a real device.

The background layers are `stack.dielectrics` boxes, as in the PDK stacks. A thin slice of oxide inside the BOX widens the layout's bounding box to the domain width, because only layers of the stack count for it. The two strips get different GDS layer numbers because the 3D mesher identifies conductors by layer number. The strips' conductivity is an effective value (`SIGMA_AL`), explained in Step 2.

In [ ]:
STRIP_WIDTH = REF["geometry"]["electrode_strip_width_um"]  # um
GAP = REF["geometry"]["electrode_gap_um"]  # um
METAL_THICKNESS = REF["geometry"]["top_metal_thickness_um"]  # um
BOX_THICKNESS = REF["geometry"]["box_thickness_um"]  # um
HANDLE_DEPTH = 100.0  # um, the handle wafer is truncated here
HALF_WIDTH = 200.0  # um, half-width of the domain
AIR_ABOVE = 100.0  # um
METAL_BOTTOM = 1.5  # um, strips' lower face above the BOX
OXIDE_OVER_METAL = 0.0  # um, no passivation over the top aluminum
SIGMA_AL_BULK = 3.77e7  # S/m, bulk aluminum
SIGMA_AL = 1.0e7  # S/m, effective: femwell's R at 10 GHz is the paper's 4.9 kohm/m
SIGMA_HANDLE = 100.0 / REF["geometry"]["handle_resistivity_ohm_cm"]  # S/m
STRIPS = {  # name: (y0, y1) in um
    "strip_g": (-GAP / 2 - STRIP_WIDTH, -GAP / 2),
    "strip_s": (GAP / 2, GAP / 2 + STRIP_WIDTH),
}
STRIP_Z = (METAL_BOTTOM, METAL_BOTTOM + METAL_THICKNESS)


def build_cps(length_um=10.0, *, lossy_metal=False, ports=False, sigma=SIGMA_AL):
    """Component and layer stack of the CPS (strips along x, ``length_um`` long).

    ``lossy_metal`` meshes the strips as regions of aluminum (for femwell);
    otherwise they are conductor layers: outlines in 2D, finite-conductivity surfaces
    in 3D. ``ports`` puts a port at each end on the signal strip. ``sigma`` is the
    strips' conductivity (S/m).
    """
    gf.gpdk.PDK.activate()
    comp = gf.Component()
    layers = {}

    def add(name, gds_layer, y0, y1, z0, z1, material, layer_type):
        comp.add_polygon(
            [(0, y0), (length_um, y0), (length_um, y1), (0, y1)], layer=gds_layer
        )
        layers[name] = Layer(
            name=name,
            gds_layer=gds_layer,
            zmin=z0,
            zmax=z1,
            thickness=z1 - z0,
            material=material,
            layer_type=layer_type,
            mesh_resolution=1.0,
        )

    add("domain", (99, 0), -HALF_WIDTH, HALF_WIDTH, -1.0, -0.9, "sio2_rf", "dielectric")
    for i, (name, span) in enumerate(STRIPS.items()):
        add(
            name,
            (10 + i, 0),
            *span,
            *STRIP_Z,
            "aluminum",
            "dielectric" if lossy_metal else "conductor",
        )
    if ports:
        for name, x, orientation in (("o1", 0.0, 180), ("o2", length_um, 0)):
            comp.add_port(
                name=name,
                center=(x, sum(STRIPS["strip_s"]) / 2),
                width=STRIP_WIDTH,
                orientation=orientation,
                port_type="electrical",
                layer=(11, 0),
            )
    z_oxide_top = STRIP_Z[1] + OXIDE_OVER_METAL
    stack = LayerStack(
        pdk_name="bj2012_cps",
        layers=layers,
        materials={
            "sio2_rf": MaterialProperties(permittivity=3.9).to_dict(),
            "handle_si": MaterialProperties(
                permittivity=11.9, conductivity=SIGMA_HANDLE
            ).to_dict(),
            "aluminum": MaterialProperties(
                permittivity=1.0, conductivity=sigma
            ).to_dict(),
            "air": MaterialProperties(permittivity=1.0).to_dict(),
        },
        dielectrics=[
            {
                "name": "handle",
                "zmin": -BOX_THICKNESS - HANDLE_DEPTH,
                "zmax": -BOX_THICKNESS,
                "material": "handle_si",
            },
            {"name": "box", "zmin": -BOX_THICKNESS, "zmax": 0.0, "material": "sio2_rf"},
            {"name": "oxide", "zmin": 0.0, "zmax": z_oxide_top, "material": "sio2_rf"},
        ],
    )
    return comp, stack


def conductors(model):
    """The signal and ground strips as the current integrals name them."""
    return (
        Conductor("strip_s", (STRIPS["strip_s"], STRIP_Z), model),
        Conductor("strip_g", (STRIPS["strip_g"], STRIP_Z), model),
    )


# Refinement box in the gap and around the strip edges: (y0, y1, z0, z1, size) um.
GAP_BOX = (-10.0, 10.0, 0.8, 5.0, 0.5)


def cross_section_sim(
    lossy_metal, output_dir, *, refined_um=1.0, metal_mesh_um=1.0, sigma=SIGMA_AL
):
    """A meshed `BoundaryModeSim` of the CPS cross-section at x = 5 um."""
    comp, stack = build_cps(lossy_metal=lossy_metal, sigma=sigma)
    for name in STRIPS:
        stack.layers[name].mesh_resolution = metal_mesh_um
    sim = BoundaryModeSim()
    sim.set_output_dir(output_dir)
    sim.set_stack(stack)
    sim.set_geometry(comp)
    sim.set_airbox(margin_x=0.0, margin_y=0.0, z_above=AIR_ABOVE, material="air")
    sim.set_cross_section("x=5")
    sim.mesh(
        preset="coarse",
        refined_mesh_size=refined_um,
        max_mesh_size=20.0,
        refinement_boxes=[GAP_BOX],
        verbose=False,
    )
    return sim, stack

## Step 1: a local reference with femwell

femwell solves the shielded cross-section at 10 GHz in about 20 seconds per case. It gives the line's index, impedance and loss without a cloud job, and it is the reference that the Palace runs are checked against. Four properties of this geometry shape the code:

- **Two modes.** The box carries two propagating modes: the line mode between the strips and a common "wall" mode with both strips against the box. The wall mode is the slower one, so the default `select_line_mode` would pick it. The line mode is the slowest one that `line_reading` does not flag as a wall mode.
- **Element order.** The contour-current impedance is not converged at order 1, so the solves use second-order elements.
- **Three cases.** The perfect-metal solve has no resistance. The two lossy solves mesh the aluminum with 0.5 µm elements, finer than the skin depth, and give $R$ and the loss: once with the effective conductivity of the Palace runs and once with bulk aluminum, so that the effect of the conductivity is on the page.
- **Impedance definitions.** The impedance depends on its definition. `z0` is $Z_{PI} = 2P/|I|^2$ with the current on the signal strip, `z_vi` is $Z_{VI} = |V|/|I|$ and `z_pv` is $Z_{PV} = |V|^2/2P$ with $V$ along the path across the gap that J1 declares (Step 3). On the perfect-metal mode they differ by 6 %; on the lossy modes they agree within 1 %.

In [ ]:
def solve_line_mode(sim, stack, model, f_hz, *, order=2, num_modes=6):
    """Solve the box modes; return ``(reading, mode, ranking)`` of the line mode.

    ``ranking`` lists every propagating mode by decreasing index as
    ``(number, n_eff, wall_mode)``: the number a Palace wave port gives it.
    """
    signal, ground = conductors(model)
    mesh = sim.mesh_path
    modes = solve_modes(
        mesh,
        epsilon=epsilon_by_region(mesh, stack, frequency_hz=f_hz),
        wavelength_um=speed_of_light / f_hz * 1e6,
        num_modes=num_modes,
        order=order,
        metallic_boundaries=True,
        n_guess=2.3,
    )
    readings = [
        (
            line_reading(
                m, frequency_hz=f_hz, mesh=mesh, signal=signal, return_=ground
            ),
            m,
        )
        for m in propagating_modes(list(modes))
    ]
    line = [rm for rm in readings if not rm[0].wall_mode]
    best = max(line, key=lambda rm: rm[0].n_eff.real)
    by_index = sorted(readings, key=lambda rm: -rm[0].n_eff.real)
    ranking = [
        (number, rm[0].n_eff, bool(rm[0].wall_mode))
        for number, rm in enumerate(by_index, start=1)
    ]
    return (*best, ranking)


def mode_power(mode):
    """Power 1/2 Re int (E x H*)_z dA of a femwell mode, in W (mesh coordinates in um)."""

    @skfem.Functional(dtype=np.complex128)
    def form(w):
        return cross(w["E"][0], np.conj(w["H"][0]))

    basis = mode.basis
    return (
        0.5
        * form.assemble(basis, E=basis.interpolate(mode.E), H=basis.interpolate(mode.H))
        * 1e-12
    )


def gap_voltage(mode, half_um, z_um, n=400):
    """V = int E_t . dl across the gap at height z_um, from y = +half_um to -half_um (E in V/m)."""
    (e_t, e_basis), _ = mode.basis.split(mode.E)
    y = half_um - 2 * half_um * (np.arange(n) + 0.5) / n
    e_y, _e_z = e_basis.interpolator(e_t)(np.array([y, np.full(n, z_um)]))
    return complex(np.sum(e_y) * (-2 * half_um * 1e-6 / n))


def impedance_definitions(mode, reading, mesh, f_hz, signal):
    """The line mode's impedance by three definitions, in ohm.

    ``z_pi`` is gsim's ``line_reading`` (2P / |I|^2 with the contour current on the signal
    strip), ``z_vi`` is |V| / |I| and ``z_pv`` is |V|^2 / 2P, with V along the path that J1
    declares: across the gap, at mid-height of the strips, a hair inside the gap faces.
    """
    inset = 1e-3 * min(GAP, METAL_THICKNESS)
    voltage = gap_voltage(mode, GAP / 2 - inset, sum(STRIP_Z) / 2)
    current = (
        electrode_current(mode, frequency_hz=f_hz, conductor=signal, mesh=mesh) * 1e-6
    )
    power = mode_power(mode)
    return {
        "z_pi": complex(reading.z0_ohm),
        "z_vi": abs(voltage) / abs(current),
        "z_pv": abs(voltage) ** 2 / (2 * power.real),
    }


femwell_results = {}
PORT_RANKING = None  # the perfect-metal ranking: what the wave port's solve sees
# label: (lossy metal?, conductivity, output folder)
FEMWELL_CASES = {
    "perfect metal": (False, SIGMA_AL, "pec"),
    "lossy, effective sigma": (True, SIGMA_AL, "lossy"),
    "lossy, bulk aluminum": (True, SIGMA_AL_BULK, "bulk"),
}
for label, (lossy, sigma_case, folder) in FEMWELL_CASES.items():
    sim_xs, stack_xs = cross_section_sim(
        lossy,
        f"./palace-sim-bj2012-xs-{folder}",
        metal_mesh_um=0.5 if lossy else 1.0,  # 0.5 um in the lossy metal
        sigma=sigma_case,
    )
    model = "volume" if lossy else "pec"
    reading, mode, ranking = solve_line_mode(sim_xs, stack_xs, model, FREQ_HZ)
    if not lossy:
        PORT_RANKING = ranking
    lp = line_params_from_neff([FREQ_HZ], [reading.n_eff], z0_ohm=[reading.z0_ohm])
    rlgc = {k: float(v[0]) for k, v in lp.rlgc.items()}
    z_defs = impedance_definitions(
        mode, reading, sim_xs.mesh_path, FREQ_HZ, conductors(model)[0]
    )
    femwell_results[label] = {
        "n": float(lp.n_rf[0]),
        "z0": complex(reading.z0_ohm),
        "z_vi": float(z_defs["z_vi"]),
        "z_pv": float(z_defs["z_pv"]),
        "alpha_db_cm": float(lp.alpha_rf_np_m[0]) * 8.686 / 100,
        **rlgc,
    }
    print(label, {k: np.round(v, 4) for k, v in femwell_results[label].items()})

Each bar divides a femwell result at 10 GHz by the paper's value, so a bar at 1 is a match. Look at the resistance bars.

In [ ]:
paper_values = {
    "Z (ohm)": (PAPER["z0_real_ohm"], "z0"),
    "L (nH/m)": (PAPER["l_nH_per_m"], "L"),
    "C (nF/m)": (PAPER["c_nF_per_m"], "C"),
    "R (kohm/m)": (PAPER["r_kohm_per_m"], "R"),
    "alpha (dB/cm)": (PAPER["alpha_dB_per_cm"], "alpha_db_cm"),
}
scale = {"z0": 1.0, "L": 1e9, "C": 1e9, "R": 1e-3, "alpha_db_cm": 1.0}
fig, ax = plt.subplots(figsize=(8, 3.8))
width = 0.26
for k, (label, res) in enumerate(femwell_results.items()):
    ratios = []
    for paper, key in paper_values.values():
        value = res[key].real if key == "z0" else res[key]
        ratios.append(value * scale[key] / paper)
    ax.bar(np.arange(len(ratios)) + (k - 1) * width, ratios, width, label=label)
ax.axhline(1.0, color="k", lw=1)
ax.set_xticks(range(len(paper_values)), list(paper_values), fontsize=8)
ax.set_ylabel("femwell / paper (10 GHz)")
ax.set_title("Metal-only CPS: local femwell reference against the paper")
ax.legend(fontsize=8)
plt.show()

With bulk aluminum the loss is half the measured one (1.83 against 3.8 dB/cm) and the resistance is 55 % below the paper's (2.19 against 4.9 k$\Omega$/m). $Z$ is 6 % and $L$ 11 % below the paper, $C$ is 4 % above. The effective conductivity brings every bar within 5 % of 1 (Step 2).

The resistance is the number that stands out. The cell below compares it with two estimates for the two strips: the DC resistance, and a uniform surface current in one skin depth around each perimeter.

In [ ]:
def skin_depth_um(sigma_s_per_m):
    """Skin depth at ``FREQ_HZ``, in um."""
    return 1e6 / np.sqrt(np.pi * FREQ_HZ * 4e-7 * np.pi * sigma_s_per_m)


SKIN_DEPTH_BULK_UM = skin_depth_um(SIGMA_AL_BULK)
SKIN_DEPTH_EFFECTIVE_UM = skin_depth_um(SIGMA_AL)
PERIMETER_UM = 2 * (STRIP_WIDTH + METAL_THICKNESS)
# Series resistance of the two strips, bulk aluminum, in kohm/m
R_DC_KOHM_M = 2e-3 / (SIGMA_AL_BULK * STRIP_WIDTH * METAL_THICKNESS * 1e-12)
R_SURFACE_KOHM_M = 2e-3 / (SIGMA_AL_BULK * SKIN_DEPTH_BULK_UM * PERIMETER_UM * 1e-12)
R_FEMWELL_KOHM_M = femwell_results["lossy, bulk aluminum"]["R"] * 1e-3

print(
    f"skin depth at {FREQ_HZ / 1e9:g} GHz: {SKIN_DEPTH_BULK_UM:.2f} um (bulk aluminum), "
    f"{SKIN_DEPTH_EFFECTIVE_UM:.2f} um (effective sigma); strip thickness "
    f"{METAL_THICKNESS:g} um"
)
print(
    f"R of the two strips, bulk aluminum: DC {R_DC_KOHM_M:.2f}, uniform surface current "
    f"{R_SURFACE_KOHM_M:.2f}, femwell {R_FEMWELL_KOHM_M:.2f} kohm/m"
)

femwell's 2.19 k$\Omega$/m is four times the DC resistance and 3.5 times the uniform surface-current estimate, because the current crowds at the gap-facing edges of the strips. It is still less than half of the measured 4.9 k$\Omega$/m.

## Step 2: the conductivity of the aluminum

To reach 4.9 k$\Omega$/m with these strips the aluminum must conduct about four times worse than bulk ($3.77\times10^7$ S/m). The notebook therefore uses an **effective conductivity of $10^7$ S/m** (10 µΩ·cm), the value for which femwell's $R$ at 10 GHz is the paper's (4.88 against 4.9 k$\Omega$/m). It stands for everything the model leaves out, such as barrier layers, roughness, edge rounding and contact resistance: it is one fitted number.

With it the second solve gives $Z = 59.4 - 5.0j\,\Omega$ against the paper's $61 - 5j\,\Omega$, $L = 436$ against 457 nH/m and $C = 0.124$ against 0.12 nF/m, each within 5 %. $L$ rises with $R$ because it contains the metal's internal inductance (408 nH/m for bulk aluminum). The loss is not an independent prediction: it is mostly $R/2Z$ and follows from the fit. The Palace runs below use the effective conductivity; they test the solver with a 3D model and a different treatment of the conductor.

The fitted value is not a measurement of the metal. A published aluminum film (Yang *et al.*, *Opt. Express* **22**(24), 29978) has a sheet resistance $R_s = 25.3$ m$\Omega$/sq (Table 1 of that paper: 25.3 ± 0.4 m$\Omega$/sq, measured). The table does not state the film thickness; taking it to be the strips' thickness $t$ is an assumption. For a film of that thickness the conductivity is

$$\sigma = \frac{1}{R_s\,t} = \frac{1}{25.3\times10^{-3}\ \Omega \times 2\times10^{-6}\ \mathrm{m}} \approx 2\times10^{7}\ \mathrm{S/m},$$

half the bulk value and twice the effective one. The cell solves the same cross-section at 10 GHz with it (same mesh and functions as above, vertical sidewalls) and prints the bulk result next to it.

In [ ]:
RS_PUBLISHED_OHM_SQ = 25.3e-3  # published sheet resistance of an Al film (Yang et al.)
SIGMA_PUBLISHED = 2.0e7  # S/m, about 1 / (R_s t)
print(
    f"1 / (R_s t) = {1 / (RS_PUBLISHED_OHM_SQ * METAL_THICKNESS * 1e-6):.3g} S/m "
    f"for R_s = {RS_PUBLISHED_OHM_SQ * 1e3:.1f} mohm/sq and t = {METAL_THICKNESS:g} um"
)
sim_s, stack_s = cross_section_sim(
    True, "./palace-sim-bj2012-xs-sigma", metal_mesh_um=0.5, sigma=SIGMA_PUBLISHED
)
reading_s, _mode_s, _ranking_s = solve_line_mode(sim_s, stack_s, "volume", FREQ_HZ)
lp_s = line_params_from_neff([FREQ_HZ], [reading_s.n_eff], z0_ohm=[reading_s.z0_ohm])
ALPHA_PUBLISHED_DB_CM = float(lp_s.alpha_rf_np_m[0]) * 8.686 / 100
R_PUBLISHED_KOHM_M = float(lp_s.rlgc["R"][0]) * 1e-3

bulk = femwell_results["lossy, bulk aluminum"]
ALPHA_BULK_DB_CM = bulk["alpha_db_cm"]
R_BULK_KOHM_M = bulk["R"] * 1e-3
print(
    f"femwell, sigma = {SIGMA_AL_BULK:.3g} S/m (bulk): {ALPHA_BULK_DB_CM:.3f} dB/cm, "
    f"R = {R_BULK_KOHM_M:.2f} kohm/m"
)
print(
    f"femwell, sigma = {SIGMA_PUBLISHED:.3g} S/m (published film): "
    f"{ALPHA_PUBLISHED_DB_CM:.3f} dB/cm, R = {R_PUBLISHED_KOHM_M:.2f} kohm/m"
)
print(f"paper: {PAPER['alpha_dB_per_cm']} dB/cm, R = {PAPER['r_kohm_per_m']} kohm/m")
gap_closed = (ALPHA_PUBLISHED_DB_CM - ALPHA_BULK_DB_CM) / (
    PAPER["alpha_dB_per_cm"] - ALPHA_BULK_DB_CM
)
print(
    f"the film closes {gap_closed:.0%} of the gap between bulk aluminum and the paper"
)

Bulk aluminum gives 1.83 dB/cm and $R = 2.19$ k$\Omega$/m; the published film gives 2.54 dB/cm and 3.17 k$\Omega$/m. The loss rises with the resistance, as $R/2Z$ does, and both stay below the measured 3.8 dB/cm and 4.9 k$\Omega$/m: the film closes about a third of the gap. About 1.3 dB/cm remains unexplained. Sloped sidewalls (vertical here), the row of vias at the edge of the gap (a plain strip here) and the measurement itself are candidates that this model does not test.

## Step 3: Palace set-up

Seven jobs cover the cross-section in 2D (`BoundaryModeSim`) and straight sections in 3D (`DrivenSim`). They are meshed and validated here, locally; nothing is submitted before the cloud runs.

| Job | What | Frequencies |
|---|---|---|
| J1 | `BoundaryModeSim`: 2D cross-section, strips as a surface condition (effective conductivity), line impedance across the gap, one job per frequency | 1, 2.5, 5, 10, 20, 40 GHz |
| J1v | as J1 with the strips **meshed as volumes of metal** (same conductivity), 0.25 µm elements in the metal | same |
| J1b | as J1 (surface condition) with bulk aluminum: the surface-condition row of the summary | 10 GHz |
| J1vb | as J1v (meshed metal) with bulk aluminum: the volume-metal row, to compare with femwell's 1.83 dB/cm | 10 GHz |
| J2 | `DrivenSim`: 200 µm straight section, wave port at each end, perfect-conductor outer wall | 1 to 40 GHz, 79 points |
| J3 | `DrivenSim`: 600 µm straight section, same wall | same |
| J4 (optional) | as J2 with the refinement near the conductors reduced to 0.6 µm | same |

### The metal as a volume

gsim has no option of that name; the route is the stack. A strip declared as a layer of type `dielectric` with a conducting material (`build_cps(lossy_metal=True)`, the femwell route above) is meshed as a region. The config writer gives that region's material a `Conductivity` entry under `Domains.Materials` and no finite-conductivity boundary on its outline; the check at the end of this step prints the result for each job. `Conductor(..., model="volume")` is the name gsim's mode tools give such a metal region.

Only the 2D jobs do this. J2 to J4 keep the strips as conductor layers: 3D shells with the finite-conductivity condition. J1v and J1vb are read for the loss through the effective index; the loss comparison below discusses what Palace returns for them. Every 2D job sets the linear solver's `mg_max_levels` to 1 (multigrid off): with multigrid on, Palace's 2D boundary-mode solve returns too much loss for a meshed metal ([awslabs/palace#1177](https://github.com/awslabs/palace/issues/1177)). The surface-condition jobs give the same result either way (J1 at 10 GHz: 3.59 dB/cm and 53.4 Ω; J1b: 1.40 dB/cm).

### The wave port's mode

The wave port solves its own 2D eigenproblem on the whole port face, with perfect conductors on the strips and on the outer wall. Two propagating modes exist there: the line mode and the common wall mode of the femwell box. Palace numbers modes by decreasing wave number. The cell below lists that ranking for the perfect-metal solve, and `WAVE_MODE` takes the line mode's number from it. The Palace results check the port's logged wave number against this solve and the de-embedded index against J1.

For coupled arms (a GSSG layout) the terminal wave ports of the [GSSG notebook](https://github.com/gdsfactory/gsim/blob/main/nbs/palace_terminal_waveport_gssg.ipynb) apply; one S-G pair needs none.

In [ ]:
print("perfect-metal modes at 10 GHz, in the order a Palace wave port numbers them:")
for number, n_eff, is_wall in PORT_RANKING:
    kind = "wall mode (both strips against the box)" if is_wall else "line mode"
    print(f"  mode {number}: n_eff = {n_eff.real:.4f}, {kind}")
PORT_LINE_MODE = next(number for number, _n, is_wall in PORT_RANKING if not is_wall)
PORT_N_EFF = {number: n_eff for number, n_eff, _w in PORT_RANKING}

The line mode is port mode 2, with $n = 2.04$. Mode 1 is the wall mode, whose index is 16 % higher ($n = 2.36$): a port left at the default mode 1 launches the wall mode, not the line.

### The outer wall in 3D

The default outer wall of `DrivenSim` is absorbing. In earlier runs of this model, an absorbing wall at ±200 µm added loss to the 3D line that J1, the same cross-section in 2D, does not have; with a perfect-conductor wall the 3D loss follows J1, and the Palace results cell prints both. The 3D runs therefore use the perfect-conductor wall that J1 and the port's eigenproblem also have. `DrivenSim` has no option for it: `PECWallDrivenSim` below sets gsim's `metallic_boundaries` switch (the one `BoundaryModeSim` has) and turns the absorbing boundary off.

### Why 200 µm and 600 µm

The 200 µm section is both the reference and the *2x-thru*: removing it from the 600 µm section leaves a 400 µm line. A 100 µm difference would give a phase difference of 2.7° and a loss difference of 0.04 dB at 10 GHz, too small to measure well; 400 µm gives 11° and 0.15 dB.

In [ ]:
BM_FREQS_GHZ = (1.0, 2.5, 5.0, 10.0, 20.0, 40.0)
LENGTHS_UM = (200, 600)
DRIVEN_FMIN, DRIVEN_FMAX, DRIVEN_POINTS = 1e9, 40e9, 79  # Hz, Hz, 0.5 GHz steps
FIT_MIN_GHZ = 5.0  # the fit and its impedance anchors start here (see Step 5)
# The line mode's number on the port face (2: mode 1 is the wall mode, see above).
WAVE_MODE = PORT_LINE_MODE
# name: (length um, refinement near conductors um)
DRIVEN_CASES = {"J2": (200, 1.0), "J3": (600, 1.0), "J4": (200, 0.6)}


# 2D jobs: name: (strips meshed as volumes of metal?, sigma S/m, GHz, output folder tag).
# J1 is the surface condition; J1v, J1vb mesh the metal (0.25 um elements in it).
BM_CASES = {
    "J1": (False, SIGMA_AL, BM_FREQS_GHZ, ""),
    "J1v": (True, SIGMA_AL, BM_FREQS_GHZ, "-vol"),
    "J1b": (False, SIGMA_AL_BULK, (10.0,), "-bulk"),
    "J1vb": (True, SIGMA_AL_BULK, (10.0,), "-volbulk"),
}

In [ ]:
def boundary_mode_sim(freq_ghz, *, volume_metal=False, sigma=SIGMA_AL, tag=""):
    """J1 family: the 2D solve at one frequency, with the voltage path across the gap.

    ``volume_metal`` meshes the strips as regions of metal of conductivity ``sigma``
    instead of leaving them as outlines with a surface condition.
    """
    sim, _stack = cross_section_sim(
        volume_metal,
        f"./palace-sim-bj2012-2d{tag}-{freq_ghz:g}ghz",
        metal_mesh_um=0.25 if volume_metal else 1.0,
        sigma=sigma,
    )
    sim.metallic_boundaries = True  # the same perfect-conductor wall as the port solve
    sim.solver.linear.mg_max_levels = 1  # multigrid off: awslabs/palace#1177
    sim.set_boundary_mode(
        freq=freq_ghz * 1e9, num_modes=4, save=0, target=2.3, tolerance=1e-8
    )
    z_mid = sum(STRIP_Z) / 2
    inset = 1e-3 * min(GAP, METAL_THICKNESS)  # a hair inside the gap, on the domain
    sim.add_impedance_path(
        "gap",
        voltage=[[GAP / 2 - inset, z_mid], [-(GAP / 2 - inset), z_mid]],
    )
    sim.write_config()
    return sim


class PECWallDrivenSim(DrivenSim):
    """A `DrivenSim` whose outer wall is a perfect conductor (J1's wall and the port solve's).

    `metallic_boundaries` is the switch that puts the outer wall under the PEC boundary
    condition; `BoundaryModeSim` has it as a field, `DrivenSim` only reads it if present.
    """

    metallic_boundaries: bool = True
    absorbing_boundary: bool = False


def driven_sim(length_um, refined_um, output_dir):
    """J2-J4: a straight section with a wave port at each end and a perfect-conductor outer wall."""
    comp, stack = build_cps(length_um, ports=True)
    sim = PECWallDrivenSim()
    sim.set_output_dir(output_dir)
    sim.set_geometry(comp)
    sim.set_stack(stack)
    sim.set_airbox(margin_x=0.0, margin_y=0.0, z_above=AIR_ABOVE, z_below=0.0)
    for port in ("o1", "o2"):
        sim.add_wave_port(
            port, layer="strip_s", max_size=True, mode=WAVE_MODE, excited=True
        )
    sim.set_driven(fmin=DRIVEN_FMIN, fmax=DRIVEN_FMAX, num_points=DRIVEN_POINTS)
    validation = sim.validate_config()
    assert validation.valid, validation.errors
    sim.mesh(
        preset="default",
        refined_mesh_size=refined_um,
        max_mesh_size=30.0,
        fmax=DRIVEN_FMAX,
        verbose=False,
    )
    sim.write_config()
    return sim


def mesh_counts(path):
    """``(tetrahedra, triangles)`` of a mesh file."""
    mesh = meshio.read(str(path))
    count = lambda kind: sum(len(c.data) for c in mesh.cells if c.type == kind)  # noqa: E731
    return count("tetra"), count("triangle")

In [ ]:
bm_sims = {ghz: boundary_mode_sim(ghz) for ghz in BM_FREQS_GHZ}
# J1v, J1b, J1vb: keyed (name, GHz)
bm_extra_sims = {
    (name, ghz): boundary_mode_sim(ghz, volume_metal=vol, sigma=sig, tag=tag)
    for name, (vol, sig, freqs, tag) in BM_CASES.items()
    if name != "J1"
    for ghz in freqs
}
driven_sims = {
    name: driven_sim(length, refined, f"./palace-sim-bj2012-{name.lower()}")
    for name, (length, refined) in DRIVEN_CASES.items()
}

In [ ]:
print("J1 (2D cross-section, same mesh at every frequency):")
n_tri = mesh_counts(bm_sims[BM_FREQS_GHZ[0]].mesh_path)[1]
print(f"  {len(bm_sims)} jobs, {n_tri} triangles each, config written for each")
for case in (c for c in BM_CASES if c != "J1"):
    sims = {g: s for (c, g), s in bm_extra_sims.items() if c == case}
    first = next(iter(sims.values()))
    config = json.loads((first.output_dir / "config.json").read_text(encoding="utf-8"))
    metal_domains = [
        m for m in config["Domains"]["Materials"] if m.get("Conductivity", 0) > 1e6
    ]
    surface_conductors = config["Boundaries"].get("Conductivity", [])
    print(
        f"{case}: {len(sims)} job(s), {mesh_counts(first.mesh_path)[1]} triangles; config: "
        f"{len(metal_domains)} metal domain(s) with Conductivity, "
        f"{len(surface_conductors)} surface-condition conductor(s); "
        f"MGMaxLevels {config['Solver']['Linear']['MGMaxLevels']}; "
        f"validate_config valid: {first.validate_config().valid}"
    )
print("3D driven sections:")
for name, sim in driven_sims.items():
    tets, tris = mesh_counts(sim.mesh_path)
    length, refined = DRIVEN_CASES[name]
    print(
        f"  {name}: {length} um, refinement {refined} um: {tets} tetrahedra, "
        f"{tris} boundary triangles, validate_config valid: {sim.validate_config().valid}"
    )

## Step 4: the cloud runs

The cell below submits J1 to J3 and the volume-metal checks J1v, J1b and J1vb (J4 too with `RUN_CONVERGENCE`) to the GDSFactory+ cloud, reuses cached results of identical jobs and waits for them. Palace does not run on every local machine, so this is a cloud step. **`RUN_CLOUD` is `False` by default**: the notebook then runs offline, and the steps below work on synthetic data and on the femwell reference. To submit only some jobs, set `CLOUD_JOBS` (for example `{"J2", "J3", ("J1", 10.0), "J1b", "J1vb"}`: five jobs); `None` submits all of them.

In [ ]:
RUN_CLOUD = False
RUN_CONVERGENCE = False
# Which jobs a cloud run submits. None: every job. To submit a few at a time, name
# them: a name ("J2", "J1b") selects the job at every frequency, a (name, GHz)
# pair ("J1", 10.0) one frequency.
CLOUD_JOBS = None  # or e.g. {"J2", "J3", ("J1", 10.0), "J1b", "J1vb"}


def selected(key):
    """True if job ``key`` is submitted: ``"J2"`` or ``(name, GHz)``."""
    if CLOUD_JOBS is None:
        return True
    name = key[0] if isinstance(key, tuple) else key
    return key in CLOUD_JOBS or name in CLOUD_JOBS


results = {}
if RUN_CLOUD:
    import gsim
    from gsim.palace.results import load_text_results

    job_ids = {}
    for ghz, sim in bm_sims.items():
        if selected(("J1", ghz)):
            job_ids["J1", ghz] = sim.run(wait=False, check_cache=True, verbose="status")
    for key, sim in bm_extra_sims.items():  # J1v, J1b and J1vb
        if selected(key):
            job_ids[key] = sim.run(wait=False, check_cache=True, verbose="status")
    for name, sim in driven_sims.items():
        if name == "J4" and not RUN_CONVERGENCE:
            continue
        if selected(name):
            job_ids[name] = sim.run(wait=False, check_cache=True, verbose="status")
    print(f"submitting {len(job_ids)} job(s): {sorted(map(str, job_ids))}")
    collected = dict(
        zip(
            job_ids,
            gsim.wait_for_results(list(job_ids.values()), verbose="status"),
            strict=True,
        )
    )
    results["J1"] = {
        ghz: load_text_results(collected["J1", ghz])
        for ghz in BM_FREQS_GHZ
        if ("J1", ghz) in collected
    }
    for case, (_vol, _sig, freqs, _tag) in BM_CASES.items():
        if case != "J1":
            results[case] = {
                ghz: load_text_results(collected[case, ghz])
                for ghz in freqs
                if (case, ghz) in collected
            }
    results["driven"] = {
        name: collected[name] for name in DRIVEN_CASES if name in collected
    }
else:
    print("RUN_CLOUD is off: nothing was submitted.")

## Step 5: de-embedding the 3D sections

The 3D sections return S-parameters. This step turns them into the line's index, loss and impedance: de-embed the short section from the long one, then fit one passive line.

**De-embedding.** Both sections are simulated with the same ports, so the end effects are the same. The 200 µm network is split into two mirrored halves with the IEEE P370 non-zero-length 2x-thru method, and removing the halves from the 600 µm network leaves a 400 µm line. For a uniform symmetric line the ABCD matrix gives the propagation constant $\gamma$ without branch ambiguity.

**Impedance.** $\gamma$ is the same for any reference impedance and any end fixture, but a wave port's S is normalized to its own mode, so `to_skrf(z0=50)` only assigns 50 Ω and the de-embedded $Z_c' = \sqrt{B/C}$ is not the line's impedance. S read in a reference $Z_{ref}$ that is not the port's gives $Z_c' = 50\,Z_c/Z_p$: S alone fixes the ratio $Z_c/Z_p = Z_c'/50$. The port mode is the perfect-metal line mode of the femwell reference, so $Z_p$ comes from femwell and $Z_c = Z_c'\,Z_p/50$ is anchored to the 3D data themselves. The Palace results check the premise: $Z_c'/50$ matches $n/n_p$, the ratio of the de-embedded index to the port's, when the metal loss leaves $C$ unchanged.

**Which $Z_p$?** The port-mode impedance depends on its definition: the femwell reference gives 58.3, 56.5 and 54.7 Ω at 10 GHz for $Z_{PI}$, $Z_{VI}$ and $Z_{PV}$. The cell below adds the metal loss to the perfect-metal line and asks which definition leaves $C = n/(cZ)$ unchanged.

In [ ]:
def capacitance_nf_per_m(result, definition):
    """C = n / (c Z) in nF/m of a femwell result, for one impedance definition."""
    z = result["z0"].real if definition == "z0" else result[definition]
    return result["n"] / (speed_of_light * z) * 1e9


for label, key in (("Z_PI", "z0"), ("Z_VI", "z_vi"), ("Z_PV", "z_pv")):
    c_perfect = capacitance_nf_per_m(femwell_results["perfect metal"], key)
    c_lossy = capacitance_nf_per_m(femwell_results["lossy, effective sigma"], key)
    print(
        f"{label}: C = {c_perfect:.4f} nF/m for perfect metal; adding the metal loss "
        f"changes it by {c_lossy / c_perfect - 1:+.1%}"
    )

$Z_{PV}$ leaves $C$ within 1 % (−0.8 %), against +2.7 % for $Z_{VI}$ and +6.3 % for $Z_{PI}$. The fit uses $Z_{PV}$, which is also the definition J1 reports. The result still moves with the choice: the Palace results print the fit for each definition, and $Z$, $L$ and $C$ change by about ±3 %, an uncertainty that no S-parameter can remove.

**J1's impedance is not the anchor.** J1 reports its own $Z_{PV}$, and the Palace results print it beside the 3D value. In the reported cloud run it is 53.4 Ω at 10 GHz, 11 % under femwell's 59.8 Ω for the same lossy line, with multigrid on or off, so the gap is not the multigrid issue; anchored on it, the fit's $Z$ followed J1 (53.1 − 4.8j Ω) and $\chi^2_{red}$ was 2.6, against 0.9 with the 3D data's own impedance. J1's index and loss serve as the comparison; its impedance is not used.

**The fit.** One passive line $Z = R_0 + R_s\sqrt{f}\,(1 + j) + j\omega L,\; Y = \omega C(\tan\delta + j)$ is fitted to $\gamma$ of the 400 µm section and the anchors, with and without the internal-reactance term, keeping the model with the lower $\chi^2$. This is the fit of the [CPW de-embedding notebook](https://github.com/gdsfactory/gsim/blob/main/nbs/palace_cpw_deembedding.ipynb), kept to one step. The functions come first.

In [ ]:
import skrf as rf
from scipy.optimize import least_squares
from skrf.calibration.deembedding import IEEEP370_SE_NZC_2xThru


def _positive_real_branch(z):
    return np.where(np.real(z) < 0, -z, z)


def extract_modal_parameters(net, length_m):
    """Zc, gamma and neff of a uniform symmetric two-port from its ABCD matrix."""
    a = net.a
    A, B, C, D = a[:, 0, 0], a[:, 0, 1], a[:, 1, 0], a[:, 1, 1]
    zc = _positive_real_branch(np.sqrt(B / C))
    exp_gl = (A + D) / 2 + B / zc
    gamma = (np.log(np.abs(exp_gl)) + 1j * np.unwrap(np.angle(exp_gl))) / length_m
    neff = np.imag(gamma) * speed_of_light / (2 * np.pi * net.f)
    return {"zc": zc, "gamma": gamma, "neff": neff}


def p370_extract(short_2xthru, long_fix_dut_fix, delta_length_m):
    """Remove the mirrored halves of the short section from the long one."""
    reference_z0 = float(np.real(np.median(short_2xthru.z0[:, 0])))
    deembedding = IEEEP370_SE_NZC_2xThru(
        dummy_2xthru=short_2xthru, z0=reference_z0, use_z_instead_ifft=True
    )
    dut = deembedding.deembed(long_fix_dut_fix)
    result = extract_modal_parameters(dut, delta_length_m)
    result["dut"] = dut
    return result


LINE_MODELS = ("sqrt_f", "skin")


def line_model(params, frequency_hz, model):
    """Series Z and shunt Y per metre: R0 + Rs sqrt(f/GHz) [(1 + j)] + jwL, wC (tan d + j)."""
    r0, rs, inductance, capacitance, tan_delta = params
    f = np.asarray(frequency_hz, float)
    omega = 2 * np.pi * f
    skin = rs * np.sqrt(f / 1e9)
    series = r0 + skin + 1j * omega * inductance
    if model == "skin":
        series = series + 1j * skin
    return series, omega * capacitance * (tan_delta + 1j)


def fit_line_parameters(
    frequency_hz,
    sections,
    anchor_frequency_hz,
    anchor_impedance_ohm,
    *,
    gamma_sigma_rel=1e-3,
    impedance_sigma_rel=1e-2,
    passive=True,
):
    """Fit one passive line to gamma of every ``(length_m, gamma)`` section and |Zc| anchors.

    gamma is unchanged by (R, L, G, C) -> (kR, kL, G/k, C/k): the anchors fix k only.
    Returns ``(best, fits)``; ``best`` has the lower reduced chi-square.
    """
    f = np.asarray(frequency_hz, float)
    lengths = np.array([length for length, _ in sections], float)
    gammas = np.array([gamma for _, gamma in sections])
    fa = np.asarray(anchor_frequency_hz, float)
    za = np.asarray(anchor_impedance_ohm, float)
    scale = np.array([100.0, 100.0, 1e-7, 1e-10, 1e-3])
    gamma_sigma = gamma_sigma_rel * np.median(np.abs(gammas))
    weight = lengths / lengths.max()  # an error in S gives d(gamma) ~ 1/length
    beta_per_omega = np.median(gammas.imag / (2 * np.pi * f))
    z_guess = float(np.median(za))
    rs_guess = 2 * z_guess * np.median(gammas.real / np.sqrt(f / 1e9))
    x0 = (
        np.array(
            [0.0, rs_guess, beta_per_omega * z_guess, beta_per_omega / z_guess, 0.0]
        )
        / scale
    )
    fits = {}
    for model in LINE_MODELS:

        def residuals(x, model=model):
            series, shunt = line_model(x * scale, f, model)
            d = (np.sqrt(series * shunt)[None] - gammas) * weight[:, None] / gamma_sigma
            za_series, za_shunt = line_model(x * scale, fa, model)
            dz = (
                np.log(np.abs(np.sqrt(za_series / za_shunt)) / za) / impedance_sigma_rel
            )
            return np.concatenate([d.real.ravel(), d.imag.ravel(), dz])

        if passive:
            # R0, Rs, L, C and tan_delta are all non-negative for a passive line. Without
            # the bound R and tan_delta trade places: both add loss, only R is large.
            res = least_squares(
                residuals,
                np.maximum(x0, 1e-6),
                method="trf",
                bounds=(0.0, np.inf),
                xtol=1e-15,
                ftol=1e-15,
                gtol=1e-15,
            )
        else:  # unbounded: a negative tan_delta then shows up instead of being clipped
            res = least_squares(
                residuals, x0, method="lm", xtol=1e-15, ftol=1e-15, gtol=1e-15
            )
        dof = max(res.fun.size - res.x.size, 1)
        names = ("R0", "Rs", "L", "C", "tan_delta")
        fits[model] = {
            "params": dict(zip(names, res.x * scale, strict=True)),
            "chi2_red": float(res.fun @ res.fun / dof),
            "model": model,
        }
    return min(fits.values(), key=lambda fit: fit["chi2_red"]), fits


def line_at(fit, f_hz):
    """gamma, Zc and RLGC of a fitted line at one frequency."""
    series, shunt = line_model(list(fit["params"].values()), [f_hz], fit["model"])
    series, shunt = series[0], shunt[0]
    omega = 2 * np.pi * f_hz
    return {
        "gamma": np.sqrt(series * shunt),
        "z0": np.sqrt(series / shunt),
        "R": series.real,
        "L": series.imag / omega,
        "G": shunt.real,
        "C": shunt.imag / omega,
    }


def summarize(line):
    """Z0, n, alpha (dB/cm) and RLGC of ``line_at``, in the paper's units."""
    gamma = line["gamma"]
    return {
        "Z0 (ohm)": line["z0"],
        "n_RF": gamma.imag / (2 * np.pi * FREQ_HZ) * speed_of_light,
        "alpha (dB/cm)": gamma.real * 8.686 / 100,
        "L (nH/m)": line["L"] * 1e9,
        "R (kohm/m)": line["R"] * 1e-3,
        "C (nF/m)": line["C"] * 1e9,
        "G (S/m)": line["G"],
    }


def anchors_in_band(anchor_frequency_hz, anchor_impedance_ohm, fit_min_ghz):
    """The impedance anchors at or above the lowest frequency of the fit."""
    fa = np.asarray(anchor_frequency_hz, float)
    za = np.asarray(anchor_impedance_ohm, float)
    keep = fa >= fit_min_ghz * 1e9 * (1 - 1e-9)
    if not keep.any():
        raise ValueError(f"no impedance anchor at or above {fit_min_ghz} GHz")
    return fa[keep], za[keep]


def deembed_and_fit(
    network_short,
    network_long,
    *,
    lengths_um=LENGTHS_UM,
    anchor_frequency_hz=None,
    anchor_impedance_ohm=None,
    fit_min_ghz=5.0,
    anchor_band_ghz=(5.0, 20.0),
    port_impedance_ohm=None,
):
    """De-embed the short section from the long one and fit the line.

    Without anchors the impedance comes from the de-embedded Zc itself, over
    ``anchor_band_ghz``. ``port_impedance_ohm`` is the impedance the ports' S refers to:
    S read in the network's reference ``z_ref`` gives ``Zc' = z_ref Zc / Zp``, so
    ``Zc = Zc' Zp / z_ref`` at about eight frequencies of the band. Without it S is taken
    to refer to the network's reference (a lumped port's S does) and the band-mean
    magnitude is the anchor.
    """
    delta_m = (lengths_um[1] - lengths_um[0]) * 1e-6
    extracted = p370_extract(network_short, network_long, delta_m)
    f = extracted["dut"].f
    mask = f >= fit_min_ghz * 1e9
    if anchor_frequency_hz is None:
        band = (f >= anchor_band_ghz[0] * 1e9) & (f <= anchor_band_ghz[1] * 1e9)
        if port_impedance_ohm is None:
            anchor_frequency_hz = f[band]
            anchor_impedance_ohm = np.full(
                band.sum(), np.abs(extracted["zc"][band]).mean()
            )
        else:
            z_ref = float(np.real(np.median(network_short.z0[:, 0])))
            in_band = np.flatnonzero(band)
            pick = in_band[
                np.unique(np.linspace(0, in_band.size - 1, 8).round().astype(int))
            ]
            anchor_frequency_hz = f[pick]
            anchor_impedance_ohm = (
                np.abs(extracted["zc"][pick]) * port_impedance_ohm / z_ref
            )
    best, fits = fit_line_parameters(
        f[mask],
        [(delta_m, extracted["gamma"][mask])],
        anchor_frequency_hz,
        anchor_impedance_ohm,
    )
    return {"extracted": extracted, "fit": best, "fits": fits, "frequency_hz": f}

## Step 6: testing the extraction on synthetic data

Before any Palace data, the chain runs on exact S-parameters of a line with the paper's metal-only values at 10 GHz, built without the extraction code: $\gamma = (R + j\omega L)/Z_0$ with $Z_0 = 61 - 5j\,\Omega$, $R = 4.9$ k$\Omega$/m and $L = 457$ nH/m gives the shunt admittance. $R$ scales as $\sqrt{f}$, $L$ and $C$ are constant and the conductance is $\omega C \tan\delta$. Each section gets a series-inductance and shunt-capacitance fixture at both ends (10 pH, 10 fF). The ABCD matrices are multiplied directly and converted to S in a real reference by the textbook formula, and noise of $10^{-3}$ is added to every S-parameter.

Three cases are recovered at 10 GHz:

- S in 50 Ω, as a lumped port gives it; the anchors come from the de-embedded $|Z_c|$.
- S in a 61 Ω reference but *assigned* 50 Ω, as a wave port's power-normalized S is, with the impedance from a "2D solve" with 1 % noise.
- The same, with the impedance from the de-embedded $|Z_c'|$ and the port's reference (61 Ω, taken as known), as for the Palace data below.

In [ ]:
FREQ_GRID_HZ = np.arange(2, 81) * 0.5e9  # 1 to 40 GHz, 79 points


def true_line(f_hz):
    """Per-metre R, L, G, C of the paper's line at ``f_hz`` (R ~ sqrt f)."""
    omega10 = 2 * np.pi * 10e9
    z0 = PAPER["z0_real_ohm"] + 1j * PAPER["z0_imag_ohm"]
    r10, inductance = PAPER["r_kohm_per_m"] * 1e3, PAPER["l_nH_per_m"] * 1e-9
    gamma10 = (r10 + 1j * omega10 * inductance) / z0
    y10 = gamma10 / z0
    capacitance, tan_delta = y10.imag / omega10, y10.real / y10.imag
    f = np.asarray(f_hz, float)
    omega = 2 * np.pi * f
    return {
        "R": r10 * np.sqrt(f / 10e9),
        "L": np.full(f.shape, inductance),
        "C": np.full(f.shape, capacitance),
        "G": omega * capacitance * tan_delta,
    }


def true_gamma_z0(f_hz):
    rlgc = true_line(f_hz)
    omega = 2 * np.pi * np.asarray(f_hz, float)
    series = rlgc["R"] + 1j * omega * rlgc["L"]
    shunt = rlgc["G"] + 1j * omega * rlgc["C"]
    return np.sqrt(series * shunt), np.sqrt(series / shunt)


def abcd_to_s(abcd, z_ref):
    """S-parameters of a two-port from its ABCD matrix, real reference ``z_ref``."""
    a, b, c, d = abcd[..., 0, 0], abcd[..., 0, 1], abcd[..., 1, 0], abcd[..., 1, 1]
    den = a + b / z_ref + c * z_ref + d
    s = np.empty(abcd.shape, dtype=complex)
    s[..., 0, 0] = (a + b / z_ref - c * z_ref - d) / den
    s[..., 0, 1] = 2 * (a * d - b * c) / den
    s[..., 1, 0] = 2 / den
    s[..., 1, 1] = (-a + b / z_ref - c * z_ref + d) / den
    return s


def synthetic_network(
    length_um, *, reference_ohm, noise, rng, fixture=(10e-12, 10e-15)
):
    """Noisy S of fixture + line + mirrored fixture, labelled with a 50 ohm reference."""
    f = FREQ_GRID_HZ
    omega = 2 * np.pi * f
    gamma, z0 = true_gamma_z0(f)
    line = section_abcd(gamma * length_um * 1e-6, z0)
    ones, zeros = np.ones_like(f), np.zeros_like(f)
    series_l = np.stack(
        [np.stack([ones, 1j * omega * fixture[0]], -1), np.stack([zeros, ones], -1)], -2
    )
    shunt_c = np.stack(
        [np.stack([ones, zeros], -1), np.stack([1j * omega * fixture[1], ones], -1)], -2
    )
    launch = series_l @ shunt_c
    launch_mirrored = shunt_c @ series_l
    s = abcd_to_s(launch @ line @ launch_mirrored, reference_ohm)
    s = s + noise * (rng.standard_normal(s.shape) + 1j * rng.standard_normal(s.shape))
    s = 0.5 * (s + np.transpose(s, (0, 2, 1)))
    return rf.Network(frequency=rf.Frequency.from_f(f, unit="Hz"), s=s, z0=50.0)


def recover(
    reference_ohm, *, noise=1e-3, seed=7, anchored=False, port_impedance_ohm=None
):
    """Run the de-embedding and fit on synthetic S; return (summary, truth summary).

    ``anchored`` takes the impedance from a noisy "2D solve"; ``port_impedance_ohm`` takes it
    from the de-embedded Zc and the port reference given (what the notebook does for Palace).
    """
    rng = np.random.default_rng(seed)
    short, long_ = (
        synthetic_network(length, reference_ohm=reference_ohm, noise=noise, rng=rng)
        for length in LENGTHS_UM
    )
    kwargs = {}
    if anchored:  # a "2D solve": |Z0| at a few frequencies, 1 % noise
        fa = np.array([5, 10, 20, 40]) * 1e9
        _, z_true = true_gamma_z0(fa)
        kwargs = {
            "anchor_frequency_hz": fa,
            "anchor_impedance_ohm": np.abs(z_true)
            * (1 + 0.01 * rng.standard_normal(fa.size)),
        }
    out = deembed_and_fit(short, long_, port_impedance_ohm=port_impedance_ohm, **kwargs)
    found = summarize(line_at(out["fit"], FREQ_HZ))
    rlgc = true_line([FREQ_HZ])
    gamma, z0 = true_gamma_z0([FREQ_HZ])
    truth = summarize(
        {
            "gamma": gamma[0],
            "z0": z0[0],
            "R": rlgc["R"][0],
            "L": rlgc["L"][0],
            "C": rlgc["C"][0],
            "G": rlgc["G"][0],
        }
    )
    return found, truth, out


synthetic = {
    "S in 50 ohm, anchors from |Zc|": recover(50.0),
    "S in 61 ohm assigned 50, 2D anchors": recover(61.0, anchored=True),
    "S in 61 ohm assigned 50, port reference 61": recover(
        61.0, port_impedance_ohm=61.0
    ),
}

In [ ]:
for case, (found, truth, out) in synthetic.items():
    print(
        f"{case}  (model: {out['fit']['model']}, chi2_red {out['fit']['chi2_red']:.2f})"
    )
    print(f"  {'quantity':14s} {'truth':>18s} {'recovered':>18s} {'error':>8s}")
    worst = 0.0
    for key, value in truth.items():
        got = found[key]
        err = abs(got - value) / abs(value)
        if key != "G (S/m)":  # G is a small correction: shown, not asserted
            worst = max(worst, err)
        print(f"  {key:14s} {value:18.4f} {got:18.4f} {err:8.2%}")
    print(f"  worst error (excluding G): {worst:.2%}")

In [ ]:
# The same test over eight noise seeds: how much of the error is noise.
stats = {case: [] for case in ("S in 50 ohm", "S in 61 ohm, 2D anchors")}
for seed in range(1, 9):
    for case, (ref, anch) in zip(stats, ((50.0, False), (61.0, True)), strict=True):
        found, truth, _out = recover(ref, seed=seed, anchored=anch)
        stats[case].append(
            {
                k: abs(found[k] - truth[k]) / abs(truth[k])
                for k in truth
                if k != "G (S/m)"
            }
        )
for case, rows in stats.items():
    print(case + ": worst / median error over 8 noise seeds")
    print(
        "  "
        + "  ".join(
            f"{k.split()[0]} {max(r[k] for r in rows):.1%}/{np.median([r[k] for r in rows]):.1%}"
            for k in rows[0]
        )
    )

With 79 frequency points the index, impedance, inductance, capacitance and loss come back within about 2 % in every seed. The resistance is the weak one: $R = \mathrm{Re}(\gamma Z_c)$ is the small real part of a mostly reactive impedance and trades places with the conductance in the fit (the fit is bounded to a passive line for that reason). With $10^{-3}$ noise on a 400 µm section it comes back typically 2 % off, and 5 % off in the worst of eight seeds. The loss in dB/cm is the better-determined number for comparing with the paper.

The plot shows the loss and the index that the de-embedding returns point by point, before the fit, for the three synthetic cases, with the truth dashed. Look at how far the points scatter around the dashed line.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
gamma_true, z_true = true_gamma_z0(FREQ_GRID_HZ)
for case, (_found, _truth, out) in synthetic.items():
    f = out["frequency_hz"] / 1e9
    axes[0].plot(f, out["extracted"]["gamma"].real * 8.686 / 100, label=case)
    axes[1].plot(
        f,
        out["extracted"]["gamma"].imag
        * speed_of_light
        / (2 * np.pi * out["frequency_hz"]),
    )
axes[0].plot(FREQ_GRID_HZ / 1e9, gamma_true.real * 8.686 / 100, "k--", label="truth")
axes[1].plot(
    FREQ_GRID_HZ / 1e9,
    gamma_true.imag * speed_of_light / (2 * np.pi * FREQ_GRID_HZ),
    "k--",
)
axes[0].set(
    xlabel="f (GHz)", ylabel="alpha (dB/cm)", title="De-embedded 400 um section"
)
axes[1].set(xlabel="f (GHz)", ylabel="n_RF", title="")
axes[0].legend(fontsize=7)
plt.show()

The extracted loss scatters around the truth by up to 12 % per frequency point from 5 to 40 GHz and by up to 24 % below 5 GHz, where the 400 µm section's loss is small. The index scatters by up to 1.2 % and 2.4 %. The scatter is the added noise. The fit averages it, which is why the recovery above reaches the loss at 10 GHz to about 1 %.

## Step 7: the Palace results

With `RUN_CLOUD` on, the next cell checks the port mode against the port log, runs the de-embedding and fit on J2 and J3 (the impedance from the de-embedded $Z_c'$ and the port-mode impedance) and prints the line next to J1's index and loss. With it off the cell says so. The line mode of each J1 solve is the mode with the largest gap voltage: on the common mode both strips sit at one potential and the gap carries almost none.

In [ ]:
def j1_line_mode(result):
    """Number of the line mode of a 2D solve: the mode with the largest gap voltage."""
    voltages = {
        m: abs(result.mode_voltage(index=1, mode=m) or 0.0) for m in result.modes
    }
    return max(voltages, key=voltages.get)

In [ ]:
palace = None
palace_summary = None
if results.get("J1") and len(results.get("driven", {})) >= 2:
    anchors = {}
    for ghz, res in results["J1"].items():
        mode = j1_line_mode(res)
        anchors[ghz] = (
            res.modes[mode]["n_eff"],
            res.characteristic_impedance(index=1, mode=mode),
        )
    fa = np.array(list(anchors)) * 1e9
    za = np.array([anchors[g][1] for g in anchors])
    fa, za = anchors_in_band(fa, za, FIT_MIN_GHZ)
    # The port solve must have carried the requested mode: its logged wave number against
    # the perfect-metal femwell solve (the same eigenproblem), and the continuity of the mode
    # over the sweep (gsim's own check of the log).
    from gsim.palace.validation import ModeExpectation, validate_solution

    k0_10 = 2 * np.pi * FREQ_HZ / speed_of_light
    for name in ("J2", "J3"):
        log_path = next(
            (
                Path(p)
                for key, p in (
                    getattr(results["driven"][name], "files", None) or {}
                ).items()
                if Path(key).name == "palace.log"
            ),
            None,
        )
        if log_path is None:
            print(
                f"{name}: no palace.log among the downloaded files, port check skipped"
            )
            continue
        report = validate_solution(
            log_path.read_text(encoding="utf-8", errors="replace"),
            mode_expectations=[
                ModeExpectation(
                    port=1,
                    mode=WAVE_MODE,
                    wave_number=complex(PORT_N_EFF[WAVE_MODE]) * k0_10,
                    wave_number_rtol=0.01,
                    frequency_ghz=FREQ_HZ / 1e9,
                )
            ],
            ports_share_cross_section=True,
        )
        for check in (*report.mode_checks, *report.port_mode_checks):
            print(f"{name} {check.name}: {check.status}; {check.evidence[0]}")
        assert not report.failures, (
            f"{name}: the wave port did not carry mode {WAVE_MODE} of the perfect-metal "
            f"solve (n = {PORT_N_EFF[WAVE_MODE].real:.4f} at 10 GHz): {report.failures}"
        )
    networks = {name: results["driven"][name].to_skrf(z0=50.0) for name in ("J2", "J3")}
    # The ports' S refers to the perfect-metal line mode (the port's own eigenproblem), not to the
    # 50 ohm that to_skrf assigns: Zc = Zc' Zp / 50 with Zp from femwell (Z_PV, see above).
    perfect = femwell_results["perfect metal"]
    palace = deembed_and_fit(
        networks["J2"],
        networks["J3"],
        port_impedance_ohm=perfect["z_pv"],
        fit_min_ghz=FIT_MIN_GHZ,
    )
    # The line must be the line mode: compare gamma with J1's complex index.
    # Above 5 GHz: the 400 um section's phase is too small below it to be
    # read to a percent. The wall mode sits well above the line mode in index.
    extracted = palace["extracted"]
    k0s = 2 * np.pi * palace["frequency_hz"] / speed_of_light
    print(
        "GHz | n_RF driven, 2D | alpha driven (raw, no fit), 2D (dB/cm) | |Zc'| x Zp/50, J1 Z_PV (ohm)"
    )
    for ghz, (n_eff, z_pv_j1) in anchors.items():
        if ghz < 5.0:
            continue
        k = int(np.argmin(np.abs(palace["frequency_hz"] - ghz * 1e9)))
        n_driven = extracted["neff"][k]
        alpha_raw = extracted["gamma"][k].real * 8.686 / 100
        alpha_2d = -n_eff.imag * k0s[k] * 8.686 / 100
        z_3d = abs(extracted["zc"][k]) * perfect["z_pv"] / 50.0
        print(
            f"{ghz:5.1f} | {n_driven:.4f}, {n_eff.real:.4f} | {alpha_raw:.3f}, {alpha_2d:.3f} | "
            f"{z_3d:.2f}, {z_pv_j1:.2f}"
        )
        assert abs(n_driven / n_eff.real - 1) < 0.04, (
            "the de-embedded index is not the line mode's: check WAVE_MODE and the "
            "port wave numbers printed above"
        )
    # The premise of the port reference: Zc'/50 = Zc/Zp should follow n/n_p (a loss that leaves
    # C unchanged). Printed, not asserted: it is a check on the story, not on the data.
    k10 = int(np.argmin(np.abs(palace["frequency_hz"] - FREQ_HZ)))
    ratio_z = abs(extracted["zc"][k10]) / 50.0
    ratio_n = extracted["neff"][k10] / PORT_N_EFF[WAVE_MODE].real
    print(
        f"10 GHz: |Zc'|/50 = {ratio_z:.3f} against n / n_port = {ratio_n:.3f} "
        f"({'consistent' if abs(ratio_z / ratio_n - 1) < 0.03 else 'CHECK the port reference'})"
    )
    palace_summary = summarize(line_at(palace["fit"], FREQ_HZ))
    print({k: np.round(v, 4) for k, v in palace_summary.items()})
else:
    print("No Palace results (RUN_CLOUD is off): nothing to extract.")

The next cell prints what the choices cost at 10 GHz: the fit for each definition of the port-mode impedance, and the fit anchored on J1's $Z_{PV}$ instead.

In [ ]:
if palace is not None:
    # How much the definition of the port impedance moves the result at 10 GHz.
    print("fit at 10 GHz for each definition of the port-mode impedance Zp:")
    for label, key in (("Z_PI", "z0"), ("Z_VI", "z_vi"), ("Z_PV", "z_pv")):
        zp = complex(perfect[key]).real
        alt = summarize(
            line_at(
                deembed_and_fit(
                    networks["J2"],
                    networks["J3"],
                    port_impedance_ohm=zp,
                    fit_min_ghz=FIT_MIN_GHZ,
                )["fit"],
                FREQ_HZ,
            )
        )
        z = alt["Z0 (ohm)"]
        print(
            f"  Zp = {label} = {zp:.2f}: Z = {z.real:.1f}{z.imag:+.1f}j ohm, "
            f"L = {alt['L (nH/m)']:.0f} nH/m, C = {alt['C (nF/m)']:.4f} nF/m, R = {alt['R (kohm/m)']:.2f} kohm/m"
        )
    # For comparison: J1's Z_PV as the anchors instead.
    by_j1 = deembed_and_fit(
        networks["J2"],
        networks["J3"],
        anchor_frequency_hz=fa,
        anchor_impedance_ohm=za,
        fit_min_ghz=FIT_MIN_GHZ,
    )
    j1_row = summarize(line_at(by_j1["fit"], FREQ_HZ))
    print(
        f"anchored on J1's Z_PV instead (chi2_red {by_j1['fit']['chi2_red']:.1f} against "
        f"{palace['fit']['chi2_red']:.1f}): Z = {j1_row['Z0 (ohm)']:.1f} ohm, "
        f"L = {j1_row['L (nH/m)']:.0f} nH/m, C = {j1_row['C (nF/m)']:.4f} nF/m"
    )

The table compares the femwell reference and the Palace fit with the paper at 10 GHz, with the relative tolerance for each quantity.

In [ ]:
paper_row = {
    "Z0 (ohm)": complex(PAPER["z0_real_ohm"], PAPER["z0_imag_ohm"]),
    "n_RF": None,
    "alpha (dB/cm)": PAPER["alpha_dB_per_cm"],
    "L (nH/m)": PAPER["l_nH_per_m"],
    "R (kohm/m)": PAPER["r_kohm_per_m"],
    "C (nF/m)": PAPER["c_nF_per_m"],
}
femwell_lossy = femwell_results["lossy, effective sigma"]
femwell_row = {
    "Z0 (ohm)": femwell_lossy["z0"],
    "n_RF": femwell_lossy["n"],
    "alpha (dB/cm)": femwell_lossy["alpha_db_cm"],
    "L (nH/m)": femwell_lossy["L"] * 1e9,
    "R (kohm/m)": femwell_lossy["R"] * 1e-3,
    "C (nF/m)": femwell_lossy["C"] * 1e9,
}
TOLERANCE = {  # relative tolerances for the metal-only line
    "Z0 (ohm)": 0.10,
    "alpha (dB/cm)": 0.30,
    "L (nH/m)": 0.10,
    "R (kohm/m)": 0.30,
    "C (nF/m)": 0.10,
}
print(
    f"{'quantity':14s} {'paper':>14s} {'femwell':>14s} {'Palace':>14s}  error / tolerance (Palace if run, else femwell)"
)
for key, paper_value in paper_row.items():
    if paper_value is None:
        continue
    fem = femwell_row[key]
    pal = palace_summary[key] if palace else None
    simulated = pal if pal is not None else fem  # Palace once it has run
    err = abs(simulated - paper_value) / abs(paper_value)
    flag = "ok" if err <= TOLERANCE[key] else "OUTSIDE"
    cells = [
        np.round(value, 3) if value is not None else "(cloud)"
        for value in (paper_value, fem, pal)
    ]
    print(
        f"{key:14s} {cells[0]!s:>14} {cells[1]!s:>14} {cells[2]!s:>14}"
        f"  {err:6.1%} / {TOLERANCE[key]:.0%} {flag}"
    )

**Reported cloud run.** The port log passes all three checks: the logged wave number is the line mode's, and the mode is continuous over the sweep and equal at the two ports. The de-embedded index follows J1's within 0.3 % from 5 to 40 GHz, and the loss from the S data directly is within 2 % of J1's (3.62 against 3.59 dB/cm at 10 GHz). The premise of the port reference holds: $|Z_c'|/50 = 1.064$ against $n/n_p = 1.062$.

The fit gives $Z = 58.1 - 5.2j\,\Omega$, $n_{RF} = 2.16$, 3.56 dB/cm, $L = 416$ nH/m, $R = 4.76$ k$\Omega$/m and $C = 0.124$ nF/m. Every quantity is within 10 % of the paper's, and the loss is 6 % below it. With the port reference taken as $Z_{PI}$, $Z_{VI}$ or $Z_{PV}$ the 3D line is $61.8 - 5.6j$, $59.9 - 5.4j$ or $58.1 - 5.2j\,\Omega$; the table reports the last.

## Step 8: the loss with the metal meshed as a volume

The loss at 10 GHz for bulk aluminum, then the same comparison over frequency for the effective conductivity that the 3D runs use. A value is *computed* when its job (J1b, J1vb, J1, J1v) has run, and otherwise *reported* from a cloud run: it is drawn with an open marker. The femwell values are always computed, including femwell on the mesh that J1vb sends to Palace.

The J1vb loss agrees with femwell, and the impedance needs care. In the cloud run stored with this notebook (Palace 0dd8d015, element order 2, multigrid off: `MGMaxLevels` 1, which every 2D job sets) J1vb gives 1.826 dB/cm, femwell 1.826 dB/cm on the same mesh: within 0.1 %. The 2.00 dB/cm of earlier runs, 9.5 % above femwell, came from a multigrid issue in Palace's 2D boundary-mode solve ([awslabs/palace#1177](https://github.com/awslabs/palace/issues/1177); a fix is pending upstream) and not from the model.

Two Palace impedances are reported for J1vb, and they differ:

- $57.26 - 2.19j\,\Omega$ is quoted from an earlier cloud run. It is within 0.2 % of femwell's $57.34 - 2.19j\,\Omega$ on the same mesh. The Palace version of that run is not recorded, so it cannot be tied to a build that has the wave-port $Z_{PV}$ fix [awslabs/palace#1119](https://github.com/awslabs/palace/pull/1119) (merged 9 Nov). It is kept as reported and is not recomputed here.
- $56.07 + 0j\,\Omega$ is the fresh cloud run (batch A, stored in `data/rf_benchmark/bj2012_electrode/`; Palace 0dd8d015, a build that predates #1119). Its $Z_{PV}$ is real-valued, so femwell's imaginary part has no counterpart, and $|Z|$ is 2.3 % below femwell's 57.38 $\Omega$ (4.4 % in the complex plane). The likely cause is the missing fix, which has not been checked. A rerun with a Palace that includes #1119 is pending; until then the two values stand side by side and neither confirms the other.

In [ ]:
def j1_loss_db_cm(result, ghz):
    """Loss (dB/cm) of the line mode of one 2D solve."""
    k0 = 2 * np.pi * ghz * 1e9 / speed_of_light
    return float(-result.modes[j1_line_mode(result)]["n_eff"].imag * k0 * 8.686 / 100)


palace_2d_loss = {  # (job, GHz) -> dB/cm, for the jobs that have run
    (case, ghz): j1_loss_db_cm(res, ghz)
    for case in BM_CASES
    for ghz, res in results.get(case, {}).items()
}
# Cloud-run values, not computed here: J1b and J1vb rerun them when RUN_CLOUD is on.
# J1vb: element order 2, multigrid off (MGMaxLevels 1, see boundary_mode_sim;
# multigrid on gave 2.00 dB/cm: palace#1177). Its Z_PV has two values: 57.26 - 2.19j is
# quoted from an earlier cloud run (Palace version not recorded); the fresh cloud run
# of batch A (Palace 0dd8d015, before awslabs/palace#1119, real-valued Z_PV) is read
# from the stored results.
REPORTED_BULK_10GHZ = {"J1b": 1.40, "J1vb": 1.826}  # dB/cm
REPORTED_J1VB_Z_PV = 57.26 - 2.19j  # ohm, J1vb, order 2, multigrid off, as reported
TWO_D_STORED = {
    (row["job"], row["freq_ghz"]): row
    for row in json.loads(
        (
            REF_PATH.parent / "rf_benchmark" / "bj2012_electrode" / "two_d_results.json"
        ).read_text(encoding="utf-8")
    )
}
CLOUD_J1VB_Z_PV = complex(*TWO_D_STORED["J1vb", 10.0]["z_pv_ohm"])  # ohm, pre-#1119

# femwell on the mesh that J1vb sends to Palace: same mesh, same metal, other solver.
_comp_vb, stack_j1vb = build_cps(lossy_metal=True, sigma=SIGMA_AL_BULK)
reading_j1vb, _mode_vb, _ranking_vb = solve_line_mode(
    bm_extra_sims["J1vb", 10.0], stack_j1vb, "volume", FREQ_HZ
)
lp_j1vb = line_params_from_neff(
    [FREQ_HZ], [reading_j1vb.n_eff], z0_ohm=[reading_j1vb.z0_ohm]
)
FEMWELL_J1VB_MESH = float(lp_j1vb.alpha_rf_np_m[0]) * 8.686 / 100  # dB/cm
FEMWELL_J1VB_MESH_Z = complex(reading_j1vb.z0_ohm)  # ohm

BULK_ROWS = (  # job, what, label of the marker
    ("J1b", "Palace, strips as a surface condition", "Palace, surface condition"),
    (
        "J1vb",
        "Palace, strips meshed as volumes of metal",
        "Palace BoundaryMode, volume metal",
    ),
)

print(f"bulk aluminum, sigma = {SIGMA_AL_BULK:.3g} S/m, 10 GHz:")
print(f"  {'paper (measured)':66s} {PAPER['alpha_dB_per_cm']:6.3f} dB/cm")
print(
    f"  {'femwell, metal meshed (computed, reference cases)':66s} "
    f"{ALPHA_BULK_DB_CM:6.3f} dB/cm"
)
print(
    f"  {'femwell, metal meshed (computed, J1vb mesh)':66s} "
    f"{FEMWELL_J1VB_MESH:6.3f} dB/cm"
)
bulk_points = {}
for job, what, label in BULK_ROWS:
    computed = (job, 10.0) in palace_2d_loss
    value = palace_2d_loss[job, 10.0] if computed else REPORTED_BULK_10GHZ[job]
    bulk_points[job] = (value, computed, label)
    status = f"computed, {job}" if computed else "reported, not run here"
    print(f"  {what + ' (' + status + ')':66s} {value:6.3f} dB/cm")
surface, meshed = bulk_points["J1b"][0], bulk_points["J1vb"][0]
shortfall = FEMWELL_J1VB_MESH - surface
print(
    f"  surface condition under femwell: {shortfall:.2f} dB/cm, "
    f"{shortfall / FEMWELL_J1VB_MESH:.0%} of femwell's total loss"
)
print(f"  Palace volume metal over femwell: {meshed / FEMWELL_J1VB_MESH - 1:+.2%}")
z_fem = FEMWELL_J1VB_MESH_Z
z_palace = REPORTED_J1VB_Z_PV
print(f"Z of the volume-metal line: femwell, J1vb mesh (computed) {z_fem:.2f} ohm")
print(
    f"  Palace J1vb, order 2 (reported) {z_palace:.2f} ohm, "
    f"{abs(z_palace - z_fem) / abs(z_fem):.1%} from femwell"
)
print(
    f"  Palace J1vb, cloud run before palace#1119 (stored) {CLOUD_J1VB_Z_PV:.2f} ohm: "
    f"|Z| {abs(CLOUD_J1VB_Z_PV) / abs(z_fem) - 1:+.1%} against femwell's |Z|"
)

In the left panel, compare each Palace marker with the femwell reference marker; in the right panel, compare the curves with the femwell and measured values at 10 GHz.

In [ ]:
fig, (ax_a, ax_b) = plt.subplots(1, 2, figsize=(10, 3.8))
ax_a.plot(
    [0], [FEMWELL_J1VB_MESH], "o", color="C0", label="femwell, metal meshed (reference)"
)
for x, (job, color) in enumerate((("J1b", "C1"), ("J1vb", "C2")), start=1):
    value, computed, label = bulk_points[job]
    ax_a.plot(
        [x], [value], "s", color=color, mfc=color if computed else "none", label=label
    )
ax_a.axhline(PAPER["alpha_dB_per_cm"], color="k", ls="--", label="paper, measured")
ax_a.set_xticks([])
ax_a.set_xlim(-0.7, 2.7)
ax_a.set_ylim(0, 5.6)
ax_a.set_ylabel("loss at 10 GHz (dB/cm)")
ax_a.set_title("Bulk aluminum: femwell and Palace")
ax_a.legend(fontsize=7, loc="upper center", ncol=2)

for job, label, color in (
    ("J1", "Palace J1, surface condition", "C1"),
    ("J1v", "Palace J1v, volume metal", "C2"),
):
    points = sorted((g, v) for (c, g), v in palace_2d_loss.items() if c == job)
    if points:
        ax_b.plot(*zip(*points, strict=True), "s-", color=color, label=label)
ax_b.plot([10], [femwell_lossy["alpha_db_cm"]], "o", color="C0", label="femwell")
ax_b.plot([10], [PAPER["alpha_dB_per_cm"]], "k*", ms=10, label="paper, measured")
if not any(c in ("J1", "J1v") for c, _g in palace_2d_loss):
    ax_b.text(
        0.03,
        0.95,
        "no Palace results (RUN_CLOUD off)",
        transform=ax_b.transAxes,
        va="top",
        fontsize=8,
    )
ax_b.set(
    xscale="log",
    xlabel="f (GHz)",
    ylabel="loss (dB/cm)",
    title=f"Effective sigma = {SIGMA_AL:.0e} S/m",
)
ax_b.legend(fontsize=7, loc="center left")
plt.show()

For bulk aluminum, the surface condition gives 1.40 dB/cm against femwell's 1.83: 0.43 dB/cm, **about 23 % of femwell's total loss** (silicon loss included; the strip is 2.4 skin depths thick). The volume-metal run agrees with femwell once multigrid is off: 1.826 against 1.826 dB/cm on the same mesh at order 2 (within 0.1 %, reported run); the 2.00 dB/cm of earlier runs came from Palace's multigrid issue ([awslabs/palace#1177](https://github.com/awslabs/palace/issues/1177)). At the effective conductivity the strip is 1.3 skin depths thick, and the reported J1 run at 10 GHz is 4 % under femwell (3.59 against 3.74 dB/cm): the shortfall depends on the conductivity.

## Step 9: the loaded line

The modulator's junction adds a series-RC branch per unit length to the shunt admittance of the metal-only line. The paper's loaded line at 10 GHz is $Z = 37 - 1j\,\Omega$, 10.8 dB/cm and $n_{RF} = 3.75$. The junction values are those of the TCAD modulator notebook at 0 V (calibrated TCAD, averaged over the striated length: $R_s = 1.66$ Ω·cm, $C_j = 0.211$ nF/m) and the paper's quoted ones (1.6 Ω·cm, 0.225 nF/m). The metal-only RLGC is the Palace fit when it exists and the femwell reference (effective conductivity) otherwise.

In [ ]:
# (R_s in ohm m, C_j in F/m) of the junction branch
JUNCTIONS = {
    "paper-quoted junction": (0.016, 0.225e-9),
    "calibrated TCAD junction": (0.0166, 0.211e-9),
}
if palace:
    line10 = line_at(palace["fit"], FREQ_HZ)
    source = "Palace"
    rlgc10 = {k: np.array([line10[k]]) for k in "RLGC"}
else:
    source = "femwell (lossy, effective sigma)"
    rlgc10 = {k: np.array([femwell_lossy[k]]) for k in "RLGC"}
for label, junction in JUNCTIONS.items():
    gamma_l, z_l = loaded_line_params([FREQ_HZ], rlgc=rlgc10, junction=junction)
    lp = line_params_from_gamma([FREQ_HZ], gamma_l, z0_ohm=z_l)
    print(
        f"{source} + {label}: Z = {z_l[0].real:.1f}{z_l[0].imag:+.1f}j ohm, "
        f"n_RF = {lp.n_rf[0]:.3f}, alpha = {lp.alpha_rf_np_m[0] * 8.686 / 100:.2f} dB/cm"
    )
LOADED = REF["loaded_line_10GHz"]
print(
    f"paper: Z = {LOADED['z0_real_ohm']:g}{LOADED['z0_imag_ohm']:+g}j ohm, "
    f"n_RF = {LOADED['n_rf']}, alpha = {LOADED['alpha_dB_per_cm']} dB/cm"
)

With the femwell metal-only line and the paper-quoted junction the loaded line is $Z = 36.0 - 0.6j\,\Omega$, $n_{RF} = 3.64$ and 10.75 dB/cm, against the paper's $37 - 1j\,\Omega$, 3.75 and 10.8 dB/cm. The calibrated TCAD junction gives $36.7 - 0.7j\,\Omega$, 3.57 and 10.32 dB/cm. In the reported cloud run, with the Palace fit as the metal-only line, the paper-quoted junction gives $35.2 - 0.7j\,\Omega$, 3.56 and 10.52 dB/cm. The effective conductivity is fitted to the paper's $R$ and the junction values come from the paper or from a TCAD calibrated to it, so this is a consistency check and not a prediction.

### Storing a cloud run

With `RUN_CLOUD` on, the next cell writes what the run produced to `data/rf_benchmark/bj2012_electrode/`: the S-parameters of the two 3D sections as Touchstone files (`J2.s2p`, `J3.s2p`, ports labelled 50 Ω as `to_skrf` assigns, see Step 5), the 2D results of every job that ran, and a `provenance.json` with the Palace version (the `Git changeset ID` line of each `palace.log`), the mesh sizes, the package versions and the date. With `RUN_CLOUD` off it does nothing.

In [ ]:
STORE_DIR = REF_PATH.parent / "rf_benchmark" / "bj2012_electrode"
if RUN_CLOUD and palace is not None:
    import importlib.metadata as metadata
    import platform
    import re
    from datetime import date

    STORE_DIR.mkdir(parents=True, exist_ok=True)
    for name, network in networks.items():  # J2.s2p and J3.s2p
        network.write_touchstone(filename=name, dir=STORE_DIR)

    def palace_version(result):
        """The ``Git changeset ID`` of a downloaded ``palace.log``, or None."""
        files = getattr(result, "files", None) or {}
        log = next(
            (Path(p) for key, p in files.items() if Path(key).name == "palace.log"),
            None,
        )
        if log is None:
            return None
        found = re.search(
            r"Git changeset ID:\s*(\S+)",
            log.read_text(encoding="utf-8", errors="replace"),
        )
        return found.group(1) if found else None

    def pair(z):
        """A complex number as [re, im] for JSON."""
        return [float(np.real(z)), float(np.imag(z))]

    two_d = []
    for case, by_ghz in results.items():
        if case == "driven":
            continue
        for ghz, res in by_ghz.items():
            mode = j1_line_mode(res)
            two_d.append(
                {
                    "job": case,
                    "freq_ghz": ghz,
                    "n_eff": pair(res.modes[mode]["n_eff"]),
                    "z_pv_ohm": pair(res.characteristic_impedance(index=1, mode=mode)),
                    "alpha_db_per_cm": j1_loss_db_cm(res, ghz),
                    "palace_version": palace_version(res),
                }
            )
    (STORE_DIR / "two_d_results.json").write_text(
        json.dumps(two_d, indent=2), encoding="utf-8"
    )
    provenance = {
        "notebook": "palace_bj2012_electrode_details.ipynb",
        "date": date.today().isoformat(),
        "jobs_submitted": sorted(str(k) for k in job_ids),
        "palace_version": {
            name: palace_version(results["driven"][name]) for name in ("J2", "J3")
        },
        "element_order": 2,
        "mg_max_levels_2d": 1,
        "tetrahedra": {
            name: mesh_counts(driven_sims[name].mesh_path)[0] for name in ("J2", "J3")
        },
        "triangles_2d": {
            "J1": mesh_counts(bm_sims[BM_FREQS_GHZ[0]].mesh_path)[1],
            "J1vb": mesh_counts(bm_extra_sims["J1vb", 10.0].mesh_path)[1],
        },
        "package_versions": {
            pkg: metadata.version(pkg)
            for pkg in ("gsim", "femwell", "scikit-fem", "gmsh", "scikit-rf")
        },
        "client": platform.platform(),
        "hardware_of_the_cloud_run": None,
        "note": "S-parameters keep the 50 ohm reference that to_skrf assigns; "
        "the port reference of the wave ports is the perfect-metal line mode (Step 5).",
    }
    (STORE_DIR / "provenance.json").write_text(
        json.dumps(provenance, indent=2), encoding="utf-8"
    )
    print(f"stored in {STORE_DIR}: {sorted(p.name for p in STORE_DIR.iterdir())}")
else:
    print("RUN_CLOUD is off: nothing to store.")

## Summary

femwell, with the metal meshed, predicts 1.83 dB/cm at 10 GHz for the paper's metal-only line with bulk aluminum: about half of the measured 3.8 dB/cm. The missing part is on the series side: the model's resistance is 2.19 k$\Omega$/m against the measured 4.9. With a conductivity fitted to the paper's resistance the same model lands within 5 % of the paper's $Z$, $L$, $C$ and loss, but only because of that fit. Palace's surface condition for the metal gives less loss than femwell, and its volume-metal solve, with multigrid off, gives the same.

| Loss at 10 GHz | dB/cm | Source |
|---|---|---|
| Paper, measured | 3.8 | paper |
| femwell, bulk aluminum ($3.77\times10^7$ S/m) | 1.83 | computed here |
| femwell, published film ($2\times10^7$ S/m) | 2.54 | computed here |
| femwell, effective conductivity ($10^7$ S/m, fitted to $R$) | 3.74 | computed here |
| Palace 2D, bulk aluminum, strips as a surface condition | 1.40 | reported cloud run; job J1b reruns it |
| Palace 2D, bulk aluminum, strips as volumes of metal | 1.83 (order 2, multigrid off) | reported cloud run; job J1vb reruns it |
| Palace 3D, effective conductivity, fit to the S-parameters | 3.56 | reported cloud run (J2, J3) |

## Limits of the comparison

- **Gap to the measurement.** The published aluminum film closes about a third of the gap (2.54 dB/cm). About 1.3 dB/cm remains unexplained; sloped sidewalls, the row of vias at the edge of the gap and the measurement itself are candidates that the model does not test.
- **The effective conductivity is a fit, not a measurement of the metal.** $Z$, $L$, $C$ and the loss that then agree with the paper are not independent of it: $L$ contains the internal inductance, which grows with $R$, and the loss is mostly $R/2Z$. The film conducts about half as well as bulk aluminum, not a quarter as well, so the fit also carries the part of $R$ that the film does not explain.
- **Surface condition and volume metal.** The surface condition is 23 % short of femwell's total loss for bulk aluminum, and that conclusion rests on femwell: Palace with the metal meshed agrees with femwell in loss (within 0.1 %) at element order 2 once multigrid is off (`MGMaxLevels` 1; the 2.00 dB/cm of earlier runs came from the multigrid issue [awslabs/palace#1177](https://github.com/awslabs/palace/issues/1177)). The impedance is not settled: the reported $57.26 - 2.19j\,\Omega$ is within 0.2 % of femwell's, but the fresh cloud run, with a Palace that predates [awslabs/palace#1119](https://github.com/awslabs/palace/pull/1119) (merged 9 Nov), gives a real $Z_{PV}$ of 56.07 $\Omega$, 2.3 % below femwell's $|Z|$, and a rerun with a Palace that has the fix is pending. The shortfall depends on the conductivity: at the effective one the reported J1 run is 4 % under femwell. The 3D runs use the surface condition too, so their agreement with J1 shows that the solvers are consistent, not that the metal loss is right.
- **The metallic box.** J1, femwell and the 3D runs share the box, so none is a check against an open device. The ±200 µm half-width is an assumption that the 3D runs do not vary.
- **Inputs read from a figure.** The electrode dimensions are read from the paper's figure. The missing oxide cover, the strips' height above the BOX, the absent top silicon and the truncated handle are assumptions.
- **Impedance definition.** The line impedance depends on how it is defined. The choice of $Z_p$ moves the 3D result by about ±3 % in $Z$, $L$ and $C$; the loss does not depend on it.
- **Silicon loss is secondary.** The perfect-metal femwell solve has 0.16 dB/cm, 4 % of the measured 3.8. The handle wafer's resistivity is the paper's.
- **The port mode is not the physical mode.** The port eigenproblem treats the strips as perfect conductors, the 3D problem as aluminum with a surface impedance, so the port's line mode ($n = 2.04$) is not the physical line's ($n = 2.22$ at 10 GHz). The 2x-thru de-embedding removes the launch mismatch; the comparison of index and impedance with J1 checks that it does.
- **Post-processing.** It is tested on synthetic S-parameters with noise. The saved notebook holds no Palace result.

## Next

- [Phase shifter of a silicon modulator](tcad_bj2012_modulator.md): the modulator's junction, which loads this line.
- [CPW length sweep and de-embedding](https://github.com/gdsfactory/gsim/blob/main/nbs/palace_cpw_deembedding.ipynb): the same 2x-thru de-embedding and fit on a CPW.

## Check against the stored results

The rows of `data/rf_benchmark/bj2012_electrode.json` that this notebook computes itself (`origin` "local") are recomputed above. The cell below compares each with its stored value and fails if one differs by more than the tolerance, so a change in femwell, gmsh or gsim that moves a published number is caught. Rows from cloud runs are not recomputed here and are listed as skipped.

In [ ]:
STORED_PATH = REF_PATH.parent / "rf_benchmark" / "bj2012_electrode.json"
STORED = json.loads(STORED_PATH.read_text(encoding="utf-8"))
NOTEBOOK_FILE = "palace_bj2012_electrode_details.ipynb"
STORED_TOLERANCE = 2e-3  # relative; femwell on one OS against the stored run


def femwell_values(prefix, res):
    """The femwell results of one case under the row ids of the stored file."""
    return {
        f"{prefix}-z0": res["z0"],
        f"{prefix}-n": res["n"],
        f"{prefix}-alpha": res["alpha_db_cm"],
        f"{prefix}-R": res["R"] * 1e-3,  # kohm/m
        f"{prefix}-L": res["L"] * 1e9,  # nH/m
        f"{prefix}-C": res["C"] * 1e9,  # nF/m
    }


computed = {
    **femwell_values("fw-bulk", femwell_results["lossy, bulk aluminum"]),
    **femwell_values("fw-eff", femwell_results["lossy, effective sigma"]),
    "fw-film-alpha": ALPHA_PUBLISHED_DB_CM,
    "fw-film-R": R_PUBLISHED_KOHM_M,
    "fw-j1vb-alpha": FEMWELL_J1VB_MESH,
    "fw-j1vb-z0": FEMWELL_J1VB_MESH_Z,
}
rlgc_femwell = {k: np.array([femwell_lossy[k]]) for k in "RLGC"}
for tag, label in (
    ("ll-paper", "paper-quoted junction"),
    ("ll-tcad", "calibrated TCAD junction"),
):
    gamma_l, z_l = loaded_line_params(
        [FREQ_HZ], rlgc=rlgc_femwell, junction=JUNCTIONS[label]
    )
    lp_l = line_params_from_gamma([FREQ_HZ], gamma_l, z0_ohm=z_l)
    computed[f"{tag}-z0"] = complex(z_l[0])
    computed[f"{tag}-n"] = float(lp_l.n_rf[0])
    computed[f"{tag}-alpha"] = float(lp_l.alpha_rf_np_m[0]) * 8.686 / 100

local_rows = [
    row
    for row in STORED["rows"]
    if row["origin"] == "local" and row["source"]["notebook"] == NOTEBOOK_FILE
]
skipped = [row["id"] for row in STORED["rows"] if row not in local_rows]
print(f"{'row':16s} {'stored':>22s} {'this run':>22s} {'rel. diff':>10s}")
worst = 0.0
for row in local_rows:
    now = complex(computed[row["id"]])  # KeyError: a row this notebook cannot recompute
    stored = complex(row["value"], row.get("value_imag") or 0.0)
    rel = abs(now - stored) / abs(stored)
    worst = max(worst, rel)
    print(
        f"{row['id']:16s} {stored.real:12.5g}{stored.imag:+9.4f}j {now.real:12.5g}{now.imag:+9.4f}j {rel:10.2e}"
    )
    assert rel <= STORED_TOLERANCE, f"{row['id']} moved by {rel:.2e}"
print(
    f"{len(local_rows)} rows recomputed, largest relative difference {worst:.1e} "
    f"(tolerance {STORED_TOLERANCE:.0e}); skipped, not computed here: {len(skipped)} rows "
    "(cloud runs)"
)

# The Palace rows quoted from earlier cloud runs against the outputs stored with the
# batch A run: the quoted values must agree with them to the digits they show.
by_id = {row["id"]: row for row in STORED["rows"]}
for row_id, job in (
    ("pal-j1vb-alpha", "J1vb"),
    ("pal-j1b-alpha", "J1b"),
    ("pal-j1-alpha", "J1"),
):
    stored_2d = TWO_D_STORED[job, 10.0]["alpha_db_per_cm"]
    rel = abs(stored_2d - by_id[row_id]["value"]) / by_id[row_id]["value"]
    assert rel <= STORED_TOLERANCE, f"{row_id} differs from the stored run by {rel:.1e}"
cloud_z = by_id["pal-j1vb-z0-cloud"]
assert abs(complex(cloud_z["value"], cloud_z["value_imag"]) - CLOUD_J1VB_Z_PV) < 1e-9
print("reported Palace losses and the cloud Z_PV agree with the stored run (two_d_results.json)")